# Image Generation — GANs Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Generator

A small DCGAN generator that takes 64-dim noise and produces a 32x32 image.

In [ ]:
```python

import torch

import torch.nn as nn

class Generator(nn.Module):

    def __init__(self, z_dim=64, img_channels=3, feat=64):

        super().__init__()

        self.net = nn.Sequential(

            nn.ConvTranspose2d(z_dim, feat * 4, kernel_size=4, stride=1, padding=0, bias=False),

            nn.BatchNorm2d(feat * 4),

            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(feat * 4, feat * 2, kernel_size=4, stride=2, padding=1, bias=False),

            nn.BatchNorm2d(feat * 2),

            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(feat * 2, feat, kernel_size=4, stride=2, padding=1, bias=False),

            nn.BatchNorm2d(feat),

            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(feat, img_channels, kernel_size=4, stride=2, padding=1, bias=False),

            nn.Tanh(),

        )

    def forward(self, z):

        return self.net(z.view(z.size(0), -1, 1, 1))

In [ ]:
```

Four transposed convs, each with `kernel_size=4, stride=2, padding=1` so they cleanly double spatial size. Output activations in [-1, 1] via tanh.

### Step 2: Discriminator

Mirror of the generator. LeakyReLU, strided convs, ends with a scalar logit.

In [ ]:
```python

class Discriminator(nn.Module):

    def __init__(self, img_channels=3, feat=64):

        super().__init__()

        self.net = nn.Sequential(

            nn.Conv2d(img_channels, feat, kernel_size=4, stride=2, padding=1),

            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat, feat * 2, kernel_size=4, stride=2, padding=1, bias=False),

            nn.BatchNorm2d(feat * 2),

            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat * 2, feat * 4, kernel_size=4, stride=2, padding=1, bias=False),

            nn.BatchNorm2d(feat * 4),

            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat * 4, 1, kernel_size=4, stride=1, padding=0),

        )

    def forward(self, x):

        return self.net(x).view(-1)

In [ ]:
```

The last conv reduces a `4x4` feature map to `1x1`. Output is a single scalar per image; apply sigmoid only during loss computation.

### Step 3: Training step

Alternate: update D once, then G once, every batch.

In [ ]:
```python

import torch.nn.functional as F

def train_step(G, D, real, z, opt_g, opt_d, device):

    real = real.to(device)

    bs = real.size(0)

    # D step

    opt_d.zero_grad()

    d_real = D(real)

    d_fake = D(G(z).detach())

    loss_d = (F.binary_cross_entropy_with_logits(d_real, torch.ones_like(d_real))

              + F.binary_cross_entropy_with_logits(d_fake, torch.zeros_like(d_fake)))

    loss_d.backward()

    opt_d.step()

    # G step

    opt_g.zero_grad()

    d_fake = D(G(z))

    loss_g = F.binary_cross_entropy_with_logits(d_fake, torch.ones_like(d_fake))

    loss_g.backward()

    opt_g.step()

    return loss_d.item(), loss_g.item()

In [ ]:
```

`G(z).detach()` in the D step is critical: we do not want gradients flowing into G during its update. Forgetting that is the classic beginner bug.

### Step 4: Full training loop on synthetic shapes

In [ ]:
```python

from torch.utils.data import DataLoader, TensorDataset

import numpy as np

def synthetic_images(num=2000, size=32, seed=0):

    rng = np.random.default_rng(seed)

    imgs = np.zeros((num, 3, size, size), dtype=np.float32) - 1.0

    for i in range(num):

        r = rng.uniform(6, 12)

        cx, cy = rng.uniform(r, size - r, size=2)

        yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")

        mask = (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2

        color = rng.uniform(-0.5, 1.0, size=3)

        for c in range(3):

            imgs[i, c][mask] = color[c]

    return torch.from_numpy(imgs)

device = "cuda" if torch.cuda.is_available() else "cpu"

data = synthetic_images()

loader = DataLoader(TensorDataset(data), batch_size=64, shuffle=True)

G = Generator(z_dim=64, img_channels=3, feat=32).to(device)

D = Discriminator(img_channels=3, feat=32).to(device)

opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

for epoch in range(10):

    for (batch,) in loader:

        z = torch.randn(batch.size(0), 64, device=device)

        ld, lg = train_step(G, D, batch, z, opt_g, opt_d, device)

    print(f"epoch {epoch}  D {ld:.3f}  G {lg:.3f}")

In [ ]:
```

`Adam(lr=2e-4, betas=(0.5, 0.999))` is the DCGAN default — the low beta1 keeps the momentum term from stabilising the adversarial game too much.

### Step 5: Sampling

In [ ]:
```python

@torch.no_grad()

def sample(G, n=16, z_dim=64, device="cpu"):

    G.eval()

    z = torch.randn(n, z_dim, device=device)

    imgs = G(z)

    imgs = (imgs + 1) / 2

    return imgs.clamp(0, 1)

In [ ]:
```

Always switch to eval mode before sampling. For DCGAN this matters because batch norm running stats are used instead of the batch's stats.

### Step 6: Spectral normalisation

A drop-in replacement for BN in the discriminator that guarantees the network is 1-Lipschitz. Fixes most "D wins too hard" failures.

In [ ]:
```python

from torch.nn.utils import spectral_norm

def build_sn_discriminator(img_channels=3, feat=64):

    return nn.Sequential(

        spectral_norm(nn.Conv2d(img_channels, feat, 4, 2, 1)),

        nn.LeakyReLU(0.2, inplace=True),

        spectral_norm(nn.Conv2d(feat, feat * 2, 4, 2, 1)),

        nn.LeakyReLU(0.2, inplace=True),

        spectral_norm(nn.Conv2d(feat * 2, feat * 4, 4, 2, 1)),

        nn.LeakyReLU(0.2, inplace=True),

        spectral_norm(nn.Conv2d(feat * 4, 1, 4, 1, 0)),

    )

In [ ]:
```

Swap `Discriminator` for `build_sn_discriminator()` and you often do not need the TTUR trick. Spectral norm is the easiest single robustness upgrade you can apply.

## Exercises

In [ ]:
1. **(Easy)** Train the DCGAN above on the synthetic circle dataset and save a grid of 16 samples at the end of each epoch. By which epoch do the generated circles become clearly circular?
2. **(Medium)** Replace the discriminator's batch norm with spectral norm. Train both versions side by side. Which one converges faster? Which one has lower variance across three seeds?
3. **(Hard)** Implement a conditional DCGAN: feed the class label into both G and D (concat one-hot to the noise in G, concat a class embedding channel in D). Train on the synthetic "circles vs squares" dataset from lesson 7 and show that class conditioning works by sampling with specific labels.